# 06 - Sparse Mixture of Experts

A sparse MoE lets a model store far more parameters than any single token pays
for. The router is twenty lines of code; its consequences reach into every
serving decision downstream.

![Sparse MoE routing](https://bankoti.github.io/llm-quest/content/images/c2/moe_router.svg)


A sparse MoE replaces one feed-forward network with a router and `E` experts.
For each token, the router computes probabilities and selects only `k` experts:

```text
p = softmax(W_router x)
w_i = p_i / sum(p_j for j in top_k(p))
y = sum(w_i * Expert_i(x)) for i in top_k(p)
```

Total parameter capacity grows with `E`, while active feed-forward compute grows
mostly with `k`. This is conditional computation, not free computation: routing,
communication, expert imbalance, memory, and capacity limits matter.

The first function uses token-choice top-k output mixing. The second implements
Switch-style execution and a balancing objective that discourages concentrated
probability on overloaded experts. Real systems differ in routing rules,
overflow handling, shared experts, and whether they drop tokens.

## Route one token by hand

Four experts, k = 2, router logits `[2.0, 1.0, 0.5, -1.0]`:

```text
softmax     -> [0.61, 0.22, 0.14, 0.03]
top-2       -> experts 0 and 1
renormalize -> [0.73, 0.27]
y = 0.73 * Expert0(x) + 0.27 * Expert1(x)
```

Experts 2 and 3 do no work for this token and receive no gradient from it.
Multiply that by millions of tokens and load balance across experts becomes a
training-stability problem, which is why the balance loss exists.

## Top-k mixing invariants

- Each token selects exactly `k` expert indices.
- Selected weights are renormalized to sum to one.
- With multiple selected experts, normalized gates provide a task-gradient path
  to the router; a normalized top-1 gate is constant, as explained below.
- A useful report includes both total and active parameter counts.

## Practice

Complete the challenge below, then force the router toward one expert and
observe the balance loss.

## Exit check

Why can two models with the same total parameter count have very different
inference cost per token?

## From output mixing to Switch execution

The first function, `route`, combines precomputed top-k outputs using normalized
selected weights. It teaches indexing. It cannot demonstrate compute savings:
those outputs have already been calculated. The second function,
`switch_forward`, must call only the experts that receive accepted inputs.

Switch uses top-1 token-choice routing. Compute softmax across all E experts and
select the largest probability. For probabilities `[0.7,0.2,0.1]`, expert 0's
output is multiplied by **0.7**, not by 1. If you normalize a singleton selection
back to 1, you remove the task gradient through that gate. Argmax itself is
discrete; the probability multiplier provides a differentiable path.

### Capacity and overflow

Our lab uses `capacity = ceil(capacity_factor * T/E)` per expert. The paper's
implementation has additional batching and layout details; this CPU exercise
uses one token group and deterministic input-order acceptance.

With T=8, E=4, factor=1, capacity is 2. Assignments `[0,0,0,0,0,1,2,3]`
overflow three tokens at expert 0. Their FFN contributions are zero. In a full
block, the residual `x + FFN(x)` preserves their input; we are not deleting
tokens from the sequence. Larger capacity costs buffers and does not fix an
expert receiving no training examples.

### A learning signal for the router

```text
f_i = number of tokens assigned to expert i / T   # before capacity dropping
P_i = mean probability of expert i across tokens
L_aux = E * sum_i(f_i * P_i)
training_loss = task_loss + coefficient * L_aux
```

At uniform assignments and probabilities, the unscaled auxiliary value is 1,
not zero. Hard assignment fractions are treated as fixed for differentiation;
P retains gradients. Do not assert that every minibatch must attain 1: the
objective is a balancing incentive, not an exact constraint. Inspect actual
loads and dropped tokens alongside it.

### Implement and falsify

1. Compute stable probabilities; use first-index argmax on ties.
2. Gather each expert's accepted input rows and call that expert once.
3. Scatter its gated outputs back into the original token positions.
4. Return outputs, a boolean acceptance mask, and the unscaled auxiliary value.
5. Check empty experts were never called and overflow inputs never ran.

The grader uses instrumented expert functions and checks nonuniform gate values.
The notebook adds autograd: compare a retained gate to the incorrect constant
gate, then train a small router with and without an auxiliary coefficient.
The expected observation is a difference in gradients and assignment behavior,
not a guaranteed quality improvement from any particular coefficient.

**Explain before moving on:** Why does lower active parameter count not guarantee
lower latency? Include dispatch, all-to-all traffic, tiny expert batches, and
load imbalance. The lab has no expert-parallel network and does not reproduce
the paper's training speedups.

Primary source: [Fedus et al., Switch Transformers, sections 2.1-2.4](https://arxiv.org/abs/2101.03961).


## Work first, then compare

CPU runtime is sufficient for the exercises. NumPy and PyTorch are available in Colab. Run the starter and grading cells after implementing the TODOs. An untouched starter is expected to fail. The reference solution below replaces the functions only when you deliberately run that cell. Then rerun the checks. No account, key, model download, or paid GPU is required.


In [ ]:
"""Sparse MoE lab.
See the paired lesson, c2/06_moe.md, for mixing versus sparse execution.
"""
import numpy as np

def softmax(x):
    e=np.exp(x-x.max(axis=-1,keepdims=True)); return e/e.sum(axis=-1,keepdims=True)

def route(expert_outputs: np.ndarray, router_logits: np.ndarray, k: int) -> np.ndarray:
    """
    expert_outputs: (tokens, experts, width)
    router_logits:  (tokens, experts)
    Returns: (tokens, width) — top-k weighted combination.
    """
    raise NotImplementedError

def switch_forward(x, router_logits, experts, capacity_factor=1.0):
    """Execute a top-1 Switch FFN on x (T,D), T>0.

    experts is a list of E callables mapping a batch (n,D) to (n,D).
    Compute softmax probabilities over ALL E experts; argmax ties pick first.
    Capacity per expert = ceil(capacity_factor*T/E), factor must be >0.
    Accept earliest assigned tokens up to capacity. Only call experts with
    accepted tokens, once per used expert. Multiply output by selected ORIGINAL
    probability (do not renormalize it to 1). Overflow output is zero: the
    surrounding Transformer residual would preserve the input.
    Return (output, accepted_mask, auxiliary_loss).
    accepted_mask has shape (T,), dtype bool.
    auxiliary_loss = E * sum(f_i * P_i), where f counts ALL pre-capacity
    assignments divided by T, and P is mean router probability per expert.
    No loss coefficient is included. Raise ValueError for nonpositive factor.
    """
    raise NotImplementedError


In [ ]:
import numpy as np
outs=np.array([[[1.,0.],[0.,2.],[9.,9.]],[[2.,2.],[4.,4.],[8.,8.]]])
logits=np.array([[2.,1.,-10.],[-10.,0.,0.]])
comb=route(outs,logits,k=2)
assert comb.shape==(2,2),f"shape: {comb.shape}"
assert np.all(comb[0]<np.array([1.01,2.01])),"expert 2 (logit -10) must be excluded"
assert np.allclose(comb[1],[6.,6.],atol=1e-5),f"equal-weight mix=[6,6], got {comb[1]}"
print("✓ shape correct")
print("✓ inactive expert excluded")
print("✓ equal-weight routing correct")
_calls = [[], [], []]
def _make_expert(i):
    def _expert(batch):
        _calls[i].append(len(batch))
        return batch * (i + 1)
    return _expert
_experts = [_make_expert(i) for i in range(3)]
_x = np.arange(1, 13, dtype=float).reshape(6, 2)
_logits = np.array([[2., 0., -1.]] * 4 + [[0., 2., -1.]] * 2)
_out, _accepted, _aux = switch_forward(_x, _logits, _experts)
assert _accepted.dtype == bool and _accepted.shape == (6,)
assert _accepted.tolist() == [True, True, False, False, True, True], 'capacity and input-order overflow'
assert _calls == [[2], [2], []], 'call only selected experts and accepted tokens'
_p = np.exp(_logits - _logits.max(axis=1, keepdims=True))
_p /= _p.sum(axis=1, keepdims=True)
np.testing.assert_allclose(_out[:2], _x[:2] * _p[:2, 0, None])
np.testing.assert_allclose(_out[2:4], 0)
np.testing.assert_allclose(_out[4:], 2 * _x[4:] * _p[4:, 1, None])
assert abs(_aux - 3 * np.dot([4/6, 2/6, 0], _p.mean(axis=0))) < 1e-10
# All ties route to expert 0; the selected gate is 1/3, not 1.
_o, _a, _l = switch_forward(np.ones((3, 2)), np.zeros((3, 3)), _experts, 3)
np.testing.assert_allclose(_o, 1/3)
assert _a.all() and abs(_l - 1) < 1e-10
try:
    switch_forward(_x, _logits, _experts, 0)
except ValueError:
    pass
else:
    raise AssertionError('nonpositive capacity must fail')
print('Switch gate, sparse execution, capacity, ties, and auxiliary loss passed.')
print("\n+200 XP — Sparse MoE complete.")


## Worked reference solution

Compare the state transitions and invariants with your implementation. Passing output tests alone does not establish memory efficiency or reproduce the paper's performance results.


In [ ]:
import math
import numpy as np

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def route(expert_outputs, router_logits, k):
    indices = np.argsort(-router_logits, axis=1, kind='stable')[:, :k]
    selected = np.take_along_axis(router_logits, indices, axis=1)
    weights = softmax(selected)
    outputs = expert_outputs[np.arange(len(indices))[:, None], indices]
    return (outputs * weights[..., None]).sum(axis=1)

def switch_forward(x, router_logits, experts, capacity_factor=1.0):
    if capacity_factor <= 0:
        raise ValueError('capacity_factor must be positive')
    t, e = router_logits.shape
    p = softmax(router_logits)
    selected = p.argmax(axis=1)
    capacity = math.ceil(capacity_factor * t / e)
    output = np.zeros_like(x, dtype=float)
    accepted = np.zeros(t, dtype=bool)
    for i, expert in enumerate(experts):
        positions = np.flatnonzero(selected == i)[:capacity]
        if len(positions):
            output[positions] = expert(x[positions]) * p[positions, i, None]
            accepted[positions] = True
    fraction = np.bincount(selected, minlength=e) / t
    auxiliary = e * np.sum(fraction * p.mean(axis=0))
    return output, accepted, float(auxiliary)


In [ ]:
import numpy as np
outs=np.array([[[1.,0.],[0.,2.],[9.,9.]],[[2.,2.],[4.,4.],[8.,8.]]])
logits=np.array([[2.,1.,-10.],[-10.,0.,0.]])
comb=route(outs,logits,k=2)
assert comb.shape==(2,2),f"shape: {comb.shape}"
assert np.all(comb[0]<np.array([1.01,2.01])),"expert 2 (logit -10) must be excluded"
assert np.allclose(comb[1],[6.,6.],atol=1e-5),f"equal-weight mix=[6,6], got {comb[1]}"
print("✓ shape correct")
print("✓ inactive expert excluded")
print("✓ equal-weight routing correct")
_calls = [[], [], []]
def _make_expert(i):
    def _expert(batch):
        _calls[i].append(len(batch))
        return batch * (i + 1)
    return _expert
_experts = [_make_expert(i) for i in range(3)]
_x = np.arange(1, 13, dtype=float).reshape(6, 2)
_logits = np.array([[2., 0., -1.]] * 4 + [[0., 2., -1.]] * 2)
_out, _accepted, _aux = switch_forward(_x, _logits, _experts)
assert _accepted.dtype == bool and _accepted.shape == (6,)
assert _accepted.tolist() == [True, True, False, False, True, True], 'capacity and input-order overflow'
assert _calls == [[2], [2], []], 'call only selected experts and accepted tokens'
_p = np.exp(_logits - _logits.max(axis=1, keepdims=True))
_p /= _p.sum(axis=1, keepdims=True)
np.testing.assert_allclose(_out[:2], _x[:2] * _p[:2, 0, None])
np.testing.assert_allclose(_out[2:4], 0)
np.testing.assert_allclose(_out[4:], 2 * _x[4:] * _p[4:, 1, None])
assert abs(_aux - 3 * np.dot([4/6, 2/6, 0], _p.mean(axis=0))) < 1e-10
# All ties route to expert 0; the selected gate is 1/3, not 1.
_o, _a, _l = switch_forward(np.ones((3, 2)), np.zeros((3, 3)), _experts, 3)
np.testing.assert_allclose(_o, 1/3)
assert _a.all() and abs(_l - 1) < 1e-10
try:
    switch_forward(_x, _logits, _experts, 0)
except ValueError:
    pass
else:
    raise AssertionError('nonpositive capacity must fail')
print('Switch gate, sparse execution, capacity, ties, and auxiliary loss passed.')
print("\n+200 XP — Sparse MoE complete.")


## Experiment and interpretation

Predict the result before running. Change one variable, repeat, and record what changed. Separate measured outcomes from assumptions.


In [ ]:
"""Autograd gate check and a tiny learned router with capacity constraints.

CPU: uv run --with torch python experiments/switch_training.py
"""
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)

class SwitchLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.router = nn.Linear(4, 4)
        self.experts = nn.ModuleList([nn.Linear(4, 4, bias=False) for _ in range(4)])
        with torch.no_grad():
            self.router.bias.copy_(torch.tensor([2., 0., 0., 0.]))

    def forward(self, x, factor=1.0):
        p = self.router(x).float().softmax(-1)
        chosen = p.argmax(-1)
        output = torch.zeros_like(x)
        accepted = torch.zeros(len(x), dtype=torch.bool)
        capacity = math.ceil(factor * len(x) / len(self.experts))
        for i, expert in enumerate(self.experts):
            idx = torch.where(chosen == i)[0][:capacity]
            if len(idx):
                output = output.index_add(0, idx, expert(x[idx]) * p[idx, i, None])
                accepted[idx] = True
        f = F.one_hot(chosen, len(self.experts)).float().mean(0)
        auxiliary = len(self.experts) * (f * p.mean(0)).sum()
        return output, auxiliary, chosen, accepted

def gate_gradient_check():
    logits = torch.tensor([[2., 1., 0.]], requires_grad=True)
    gate = logits.softmax(-1).max(-1).values
    correct = torch.autograd.grad((gate * 3).sum(), logits, retain_graph=True)[0]
    incorrect = torch.autograd.grad(((gate / gate) * 3).sum(), logits)[0]
    assert correct.abs().sum() > 0
    torch.testing.assert_close(incorrect, torch.zeros_like(incorrect))
    return {'retained_gate_gradient': correct.tolist(), 'renormalized_gate_gradient': incorrect.tolist()}

def train_router():
    rows = []
    for coefficient in [0., .1, 1.]:
        torch.manual_seed(71)
        x = torch.randn(32, 4)
        target = x.tanh()
        model = SwitchLayer()
        optimizer = torch.optim.Adam(model.parameters(), lr=.02)
        for step in range(100):
            output, auxiliary, choices, accepted = model(x)
            task = F.mse_loss(output, target)
            optimizer.zero_grad()
            (task + coefficient * auxiliary).backward()
            optimizer.step()
        with torch.no_grad():
            output, auxiliary, choices, accepted = model(x)
            rows.append({'coefficient': coefficient, 'task_loss': F.mse_loss(output,target).item(),
                         'auxiliary': auxiliary.item(), 'loads': torch.bincount(choices,minlength=4).tolist(),
                         'dropped': int((~accepted).sum())})
    return rows

if __name__ == '__main__':
    print(json.dumps(gate_gradient_check(), indent=2))
    print(json.dumps(train_router(), indent=2))
    print('One synthetic batch and seed, not a quality benchmark. Compare load, overflow, and task loss; balancing is not a guaranteed improvement.')
